In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-31T13:22:34.573758Z', 'done': True, 'done_reason': 'stop', 'total_duration': 215740375, 'load_duration': 112975125, 'prompt_eval_count': 32, 'prompt_eval_duration': 21059375, 'eval_count': 8, 'eval_duration': 76783459, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019d440f-6af4-7af1-b694-c706e017e90a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

## StrOutputParser

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template="What is the capital of {country}? Return the name of the city only",
    input_variables=["country"],
)

prompt = prompt_template.invoke({"country": "France"})

print(prompt)

ai_message = llm.invoke(prompt_template.invoke({"country": "France"}))

print(ai_message)

output_parser = StrOutputParser()

answer = output_parser.invoke(llm.invoke(prompt_template.invoke({"country": "France"})))

text='What is the capital of France? Return the name of the city only'
content='Paris' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-31T13:22:37.001563Z', 'done': True, 'done_reason': 'stop', 'total_duration': 180727708, 'load_duration': 89899708, 'prompt_eval_count': 39, 'prompt_eval_duration': 76816625, 'eval_count': 2, 'eval_duration': 11486709, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'} id='lc_run--019d440f-7491-78e3-afac-c477ad5d0446-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 2, 'total_tokens': 41}


In [13]:
ai_message

AIMessage(content='Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-29T10:27:27.973169Z', 'done': True, 'done_reason': 'stop', 'total_duration': 410026625, 'load_duration': 124092416, 'prompt_eval_count': 39, 'prompt_eval_duration': 259093083, 'eval_count': 3, 'eval_duration': 23268708, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019d3922-60c6-7523-b86e-9f5474e692f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 3, 'total_tokens': 42})

In [14]:
answer

'Paris.'

## BaseModel

In [11]:
from pydantic import BaseModel, Field

# PromptTemplate 대체
# 모델 설정
class CountryDetail(BaseModel):
    # 타입 설정
    capital: str = Field(description="The capital city of the country")
    population: int = Field(description="The population of the country")
    language: str = Field(description="The official language of the country")
    currency: str = Field(description="The currency used in the country")

structured_llm = llm.with_structured_output(CountryDetail)

In [12]:
country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}:
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format. and return the JSON dictionary only
    """,
    input_variables=["country"],
)

json_ai_message = structured_llm.invoke(country_detail_prompt.invoke({"country": "France"}))

In [13]:
(json_ai_message)

CountryDetail(capital='Paris', population=67, language='French', currency='Euro')

In [24]:
json_ai_message.model_dump()

{'capital': 'Paris',
 'population': 67,
 'language': 'French',
 'currency': 'Euro'}

In [23]:
json_ai_message.model_dump()['capital']

'Paris'

## JsonOutputParser

In [25]:
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}:
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format. and return the JSON dictionary only
    """,
    input_variables=["country"],
)

country_detail_prompt.invoke({"country": "France"})

output_parser = JsonOutputParser()
json_ai_message = llm.invoke(country_detail_prompt.invoke({"country": "France"}))
output_parser.invoke(json_ai_message)

OutputParserException: Invalid json output: ```
{
    "capital": "Paris",
    "population": 67_000_000,
    "language": "French",
    "currency": "Euro"
}
```
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [28]:
json_ai_message.content

'```\n{\n    "capital": "Paris",\n    "population": 67_000_000,\n    "language": "French",\n    "currency": "Euro"\n}\n```'

In [30]:
type(json_ai_message.content)

str